<a href="https://colab.research.google.com/github/gusbrz/Atrio/blob/main/tokens_previsibilidade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pymupdf transformers torch

import re
import random
import unicodedata

import fitz  # PyMuPDF
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokener = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()
print(f"Modelo carregado em: {device}")

STOPWORDS = {
    "para", "como", "mais", "pelo", "pela", "pelos", "pelas", "essa", "esse",
    "essas", "esses", "esta", "este", "estas", "estes", "isso", "isto",
    "aquele", "aquela", "aqueles", "aquelas", "entre", "sobre", "após", "até",
    "sendo", "seus", "suas", "onde", "quando", "porque", "pois", "também",
    "muito", "muita", "muitos", "muitas", "cada", "outro", "outra", "outros",
    "outras", "ainda", "mesmo", "mesma", "depois", "antes", "desde", "contra",
    "dele", "dela", "deles", "delas", "nosso", "nossa", "foram", "será",
    "está", "estão", "tinha", "havia", "seja", "sido", "pode", "podem",
    "numa", "num", "nas", "nos", "dos", "das", "com", "que", "uma", "uns",
    "umas", "não", "mas", "por", "sem", "sob", "ele", "ela", "eles", "elas",
}

def extrair_e_limpar_pdf(caminho_pdf):
    """Extrai texto do PDF, junta palavras hifenizadas e normaliza espaços."""
    with fitz.open(caminho_pdf) as doc:
        paginas = [pagina.get_text("text") for pagina in doc]

    texto = "\n".join(paginas)
    texto = unicodedata.normalize("NFKC", texto)
    # "opera-\ncional" -> "operacional" (antes de colapsar os espaços)
    texto = re.sub(r"(\w)[-\u00ad]\s*\n\s*(\w)", r"\1\2", texto)
    texto = texto.replace("\u00ad", "")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

# prefixo (pontuação) + núcleo (letras/dígitos/_ e hífens internos) + sufixo (pontuação)
_TOKEN_RE = re.compile(r"^(\W*)(\w[\w\-]*\w|\w)(\W*)$")


def _id_no_vocab(token):
    """Retorna o id do token no vocabulário (tenta como está e em minúsculas)."""
    for t in (token, token.lower()):
        tid = tokener.convert_tokens_to_ids(t)
        if tid != tokener.unk_token_id:
            return tid
    return None


def substituir_tokens_previsiveis_bert(
    texto,
    limiar_probabilidade=0.65,
    top_k=5,
    taxa_substituicao=0.25,
    palavras_alvo=None,
    seed=None,
):
    """
    Usa BERT (MLM) para identificar palavras altamente previsíveis (ou palavras-alvo)
    e substituí-las por alternativas contextuais menos óbvias (rank 2 até top_k).

    Limitação: cada palavra é substituída por UM token do vocabulário; palavras
    que o BERT quebra em subwords têm prob_original = 0 e só são trocadas se
    estiverem em palavras_alvo.
    """
    if seed is not None:
        random.seed(seed)

    alvos = {p.lower() for p in palavras_alvo} if palavras_alvo else set()
    sentencas = re.split(r"(?<=[.!?])\s+", texto)
    resultado = []

    for sentenca in sentencas:
        palavras = sentenca.split()
        if len(palavras) < 4:
            resultado.append(sentenca)
            continue

        novas = palavras.copy()

        for i, palavra in enumerate(palavras):
            m = _TOKEN_RE.match(palavra)
            if not m:
                continue
            prefixo, nucleo, sufixo = m.groups()

            # Ignora curtas, com dígitos/hífen, siglas, nomes próprios no meio e stopwords
            if len(nucleo) <= 3 or not nucleo.isalpha() or nucleo.isupper():
                continue
            if i > 0 and nucleo[0].isupper():
                continue
            if nucleo.lower() in STOPWORDS:
                continue

            eh_alvo = nucleo.lower() in alvos
            if not eh_alvo and random.random() > taxa_substituicao:
                continue

            # Frase mascarada (mantém as trocas já feitas nas palavras anteriores)
            temp = novas.copy()
            temp[i] = prefixo + tokener.mask_token + sufixo
            frase_mascarada = " ".join(temp)

            inputs = tokener(
                frase_mascarada,
                return_tensors="pt",
                truncation=True,
                max_length=512,
            ).to(device)

            mask_pos = torch.where(inputs["input_ids"][0] == tokener.mask_token_id)[0]
            if len(mask_pos) == 0:  # máscara cortada pela truncagem
                continue

            with torch.inference_mode():
                logits = model(**inputs).logits[0, mask_pos[0], :]
                probs = torch.softmax(logits, dim=-1)

            id_original = _id_no_vocab(nucleo)
            prob_original = probs[id_original].item() if id_original is not None else 0.0

            if not (eh_alvo or prob_original >= limiar_probabilidade):
                continue

            top_ids = torch.topk(probs, top_k + 10).indices.tolist()
            # Descarta o top-1 do modelo (a escolha mais óbvia), se não for a própria original
            if top_ids and top_ids[0] != id_original:
                top_ids = top_ids[1:]

            termina_s = nucleo.lower().endswith("s")
            candidatos = []
            for tid in top_ids:
                tok = tokener.convert_ids_to_tokens(tid)
                if (
                    tok.startswith("##")
                    or not tok.isalpha()
                    or len(tok) <= 2
                    or tok.lower() == nucleo.lower()
                    or tok.lower() in STOPWORDS
                    or tok.endswith("s") != termina_s  # concordância de número
                ):
                    continue
                candidatos.append(tok)

            if not candidatos:
                continue

            escolhido = random.choice(candidatos[: max(1, top_k - 1)])
            if nucleo[0].isupper():
                escolhido = escolhido.capitalize()
            novas[i] = prefixo + escolhido + sufixo

        resultado.append(" ".join(novas))

    return " ".join(resultado)

texto_teste = (
    "A inteligência artificial desempenha um papel fundamental na transformação "
    "digital das empresas modernas, promovendo eficiência operacional e inovação constante."
)

texto_reescrito = substituir_tokens_previsiveis_bert(
    texto_teste,
    limiar_probabilidade=0.40,  # Quanto menor, mais palavras serão trocadas
    top_k=5,
    taxa_substituicao=0.50,
    seed=42,
)

print("Original:\n", texto_teste)
print("\nSem tokenização previsível (BERT):\n", texto_reescrito)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado em: cpu
Original:
 A inteligência artificial desempenha um papel fundamental na transformação digital das empresas modernas, promovendo eficiência operacional e inovação constante.

Sem tokenização previsível (BERT):
 A inteligência estratégica assume um papel fundamental na transformação digital das organizações modernas, promovendo eficiência operacional e inovação constante.


In [1]:
!pip install -q pymupdf transformers torch

import re
import random
import unicodedata
import fitz  # PyMuPDF
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Modelo BERTimbau treinado em corpus jurídico brasileiro
MODEL_NAME = "rufimelo/Legal-BERTimbau-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()
print(f"Modelo jurídico carregado em: {device}")

STOPWORDS = {
    "para", "como", "mais", "pelo", "pela", "pelos", "pelas", "essa", "esse",
    "essas", "esses", "esta", "este", "estas", "estes", "isso", "isto",
    "aquele", "aquela", "aqueles", "aquelas", "entre", "sobre", "após", "até",
    "sendo", "seus", "suas", "onde", "quando", "porque", "pois", "também",
    "muito", "muita", "muitos", "muitas", "cada", "outro", "outra", "outros",
    "outras", "ainda", "mesmo", "mesma", "depois", "antes", "desde", "contra",
    "dele", "dela", "deles", "delas", "nosso", "nossa", "foram", "será",
    "está", "estão", "tinha", "havia", "seja", "sido", "pode", "podem",
    "numa", "num", "nas", "nos", "dos", "das", "com", "que", "uma", "uns",
    "umas", "não", "mas", "por", "sem", "sob", "ele", "ela", "eles", "elas",
}

# Termos técnicos que NÃO devem ser substituídos para preservar precisão jurídica
TERMOS_PROTEGIDOS = {
    "acórdão", "agravo", "apelação", "autos", "citação", "competência",
    "decadência", "embargos", "execução", "habeas", "impetrante", "inicial",
    "intimação", "jurisprudência", "liminar", "litisconsórcio", "mandado",
    "mérito", "nulidade", "ônus", "petição", "prescrição", "preclusão",
    "recurso", "réu", "sentença", "súmula", "trânsito", "tutela", "juízo",
}

# Equivalências diretas para elevar o registro de termos comuns/previsíveis
GLOSSARIO_JURIDICO = {
    "acordo": ["avença", "pacto", "ajuste"],
    "acordos": ["avenças", "pactos", "ajustes"],
    "contrato": ["instrumento", "avença", "pacto"],
    "contratos": ["instrumentos", "avenças", "pactos"],
    "decisão": ["provimento", "deliberação", "pronunciamento"],
    "decisões": ["provimentos", "deliberações", "pronunciamentos"],
    "direito": ["prerrogativa", "faculdade"],
    "direitos": ["prerrogativas", "faculdades"],
    "empresa": ["sociedade", "entidade"],
    "empresas": ["sociedades", "entidades"],
    "fundamental": ["basilar", "essencial", "primordial"],
    "importante": ["relevante", "substancial", "imprescindível"],
    "lei": ["diploma", "norma", "preceito"],
    "leis": ["diplomas", "normas", "preceitos"],
    "mudança": ["alteração", "modificação", "mutação"],
    "obrigação": ["encargo", "dever", "múnus"],
    "obrigações": ["encargos", "deveres"],
    "papel": ["função", "atribuição", "mister"],
    "pedido": ["pleito", "requerimento", "postulação"],
    "pedidos": ["pleitos", "requerimentos", "postulações"],
    "pessoa": ["jurisdicionado", "partícipe", "sujeito"],
    "processo": ["feito", "demanda", "procedimento"],
    "processos": ["feitos", "demandas", "procedimentos"],
    "prova": ["elemento", "lastro", "indício"],
    "provas": ["elementos", "indícios"],
    "regra": ["preceito", "diretriz", "norma"],
    "regras": ["preceitos", "diretrizes", "normas"],
    "resultado": ["desfecho", "corolário", "efeito"],
}

# Conjunto auxiliar para dar preferência a candidatos jurídicos sugeridos pelo BERT
VOCABULARIO_JURIDICO = {
    termo for lista in GLOSSARIO_JURIDICO.values() for termo in lista
} | {
    "alegações", "amparo", "aplicação", "atribuição", "cabimento", "cláusula",
    "comprovação", "concessão", "conduta", "conformidade", "constituição",
    "cumprimento", "dano", "dever", "dispositivo", "eficácia", "entendimento",
    "exigência", "fundamento", "garantia", "hipótese", "ilicitude", "incidência",
    "indeferimento", "interesse", "interpretação", "legitimidade", "manifestação",
    "medida", "natureza", "normativo", "observância", "ocorrência", "ordem",
    "parâmetro", "prejuízo", "pressuposto", "princípio", "procedência",
    "reparação", "requisito", "responsabilidade", "restituição", "sanção",
    "subsídio", "sustentação", "teor", "validade", "vedação", "vínculo",
}

def extrair_e_limpar_pdf(caminho_pdf):
    """Extrai texto do PDF, junta palavras hifenizadas e normaliza espaços."""
    with fitz.open(caminho_pdf) as doc:
        paginas = [pagina.get_text("text") for pagina in doc]
    texto = "\n".join(paginas)
    texto = unicodedata.normalize("NFKC", texto)
    texto = re.sub(r"(\w)[-\u00ad]\s*\n\s*(\w)", r"\1\2", texto)
    texto = texto.replace("\u00ad", "")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

_TOKEN_RE = re.compile(r"^(\W*)(\w[\w\-]*\w|\w)(\W*)$")

def _id_no_vocab(token):
    for t in (token, token.lower()):
        tid = tokenizer.convert_tokens_to_ids(t)
        if tid != tokenizer.unk_token_id:
            return tid
    return None

def substituir_tokens_previsiveis_juridico(
    texto,
    limiar_probabilidade=0.45,
    top_k=6,
    taxa_substituicao=0.35,
    palavras_alvo=None,
    usar_glossario=True,
    seed=None,
):
    """
    Remove a tokenização previsível usando Legal-BERTimbau + vocabulário jurídico,
    preservando termos técnicos protegidos.
    """
    if seed is not None:
        random.seed(seed)

    alvos = {p.lower() for p in palavras_alvo} if palavras_alvo else set()
    sentencas = re.split(r"(?<=[.!?])\s+", texto)
    resultado = []

    for sentenca in sentencas:
        palavras = sentenca.split()
        if len(palavras) < 4:
            resultado.append(sentenca)
            continue

        novas = palavras.copy()

        for i, palavra in enumerate(palavras):
            m = _TOKEN_RE.match(palavra)
            if not m:
                continue
            prefixo, nucleo, sufixo = m.groups()
            nucleo_lower = nucleo.lower()

            # Ignora curtas, siglas, nomes próprios no meio, stopwords e termos protegidos
            if len(nucleo) <= 3 or not nucleo.isalpha() or nucleo.isupper():
                continue
            if i > 0 and nucleo[0].isupper():
                continue
            if nucleo_lower in STOPWORDS or nucleo_lower in TERMOS_PROTEGIDOS:
                continue

            eh_alvo = nucleo_lower in alvos
            em_glossario = usar_glossario and nucleo_lower in GLOSSARIO_JURIDICO

            if not (eh_alvo or em_glossario) and random.random() > taxa_substituicao:
                continue

            # 1) Substituição direta pelo glossário jurídico (quando aplicável)
            if em_glossario:
                escolhido = random.choice(GLOSSARIO_JURIDICO[nucleo_lower])
                if nucleo[0].isupper():
                    escolhido = escolhido.capitalize()
                novas[i] = prefixo + escolhido + sufixo
                continue

            # 2) Substituição contextual via Legal-BERTimbau (MLM)
            temp = novas.copy()
            temp[i] = prefixo + tokenizer.mask_token + sufixo
            frase_mascarada = " ".join(temp)

            inputs = tokenizer(
                frase_mascarada,
                return_tensors="pt",
                truncation=True,
                max_length=512,
            ).to(device)

            mask_pos = torch.where(inputs["input_ids"][0] == tokenizer.mask_token_id)[0]
            if len(mask_pos) == 0:
                continue

            with torch.inference_mode():
                logits = model(**inputs).logits[0, mask_pos[0], :]
                probs = torch.softmax(logits, dim=-1)

            id_original = _id_no_vocab(nucleo)
            prob_original = probs[id_original].item() if id_original is not None else 0.0

            if not (eh_alvo or prob_original >= limiar_probabilidade):
                continue

            top_ids = torch.topk(probs, top_k + 12).indices.tolist()
            if top_ids and top_ids[0] != id_original:
                top_ids = top_ids[1:]

            termina_s = nucleo_lower.endswith("s")
            candidatos_juridicos = []
            candidatos_gerais = []

            for tid in top_ids:
                tok = tokenizer.convert_ids_to_tokens(tid)
                if (
                    tok.startswith("##")
                    or not tok.isalpha()
                    or len(tok) <= 2
                    or tok.lower() == nucleo_lower
                    or tok.lower() in STOPWORDS
                    or tok.endswith("s") != termina_s
                ):
                    continue

                if tok.lower() in VOCABULARIO_JURIDICO:
                    candidatos_juridicos.append(tok)
                else:
                    candidatos_gerais.append(tok)

            # Prioriza candidatos jurídicos sugeridos pelo modelo; senão usa os gerais
            pool = candidatos_juridicos if candidatos_juridicos else candidatos_gerais[: max(1, top_k - 1)]
            if not pool:
                continue

            escolhido = random.choice(pool)
            if nucleo[0].isupper():
                escolhido = escolhido.capitalize()
            novas[i] = prefixo + escolhido + sufixo

        resultado.append(" ".join(novas))

    return " ".join(resultado)


# --- Teste ---
texto_teste = (
    "A empresa descumpriu o contrato e ignorou as regras previstas na lei, "
    "motivo pelo qual o pedido de indenização tem papel fundamental neste processo."
)

texto_reescrito = substituir_tokens_previsiveis_juridico(
    texto_teste,
    limiar_probabilidade=0.40,
    top_k=6,
    taxa_substituicao=0.50,
    usar_glossario=True,
    seed=42,
)

print("Original:\n", texto_teste)
print("\nCom linguajar técnico jurídico (Legal-BERTimbau):\n", texto_reescrito)

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


config.json:   0%|          | 0.00/863 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

... (2 outputs left out)

Modelo jurídico carregado em: cpu


model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Original:
 A empresa descumpriu o contrato e ignorou as regras previstas na lei, motivo pelo qual o pedido de indenização tem papel fundamental neste processo.

Com linguajar técnico jurídico (Legal-BERTimbau):
 A sociedade descumpriu o avença e ignorou as preceitos previstas na lei, motivo pelo qual o pleito de indenização tem função basilar naquele procedimento.
